# Stable Baselines3

[Stable Baselines3 (SB3)](https://stable-baselines3.readthedocs.io/en/master/) es una librería de algoritmos de aprendizaje por refuerzo (RL) construida sobre PyTorch. Ofrece implementaciones probadas, optimizadas y testeadas de los algoritmos vistos en clase (DQN, PPO, A2C, SAC, ...), lo que permite entrenar agentes sin reimplementar todo desde cero.

El objetivo de esta notebook es:

1. Entrenar un agente con **DQN** sobre `LunarLander` y recorrer el flujo completo: entorno → entrenamiento → evaluación → video.
2. Comprender **qué algoritmos hay disponibles y cómo elegir** el adecuado según el problema.
3. Dejar planteadas varias líneas de **exploración** (otros algoritmos, control continuo, callbacks, ajuste de hiperparámetros, ...).

SB3 expone la **misma API para todos los algoritmos**:

```python
model = Algoritmo(policy, env)   # crear
model.learn(total_timesteps)     # entrenar
action, _ = model.predict(obs)   # usar
```

En consecuencia, cambiar de algoritmo suele implicar modificar una sola línea.

## El ecosistema

- **[Stable-Baselines3](https://stable-baselines3.readthedocs.io/)**: los algoritmos principales (DQN, PPO, A2C, SAC, TD3, DDPG).
- **[SB3-Contrib](https://sb3-contrib.readthedocs.io/)**: algoritmos más experimentales o especializados (QR-DQN, TQC, RecurrentPPO, MaskablePPO, TRPO, ...).
- **[RL Baselines3 Zoo](https://rl-baselines3-zoo.readthedocs.io/)**: scripts de entrenamiento, hiperparámetros ya ajustados para decenas de entornos y herramientas de tuning con Optuna.

## Instalación

- Paquete de pip: https://pypi.org/project/stable-baselines3
- Paquete de Conda: https://anaconda.org/conda-forge/stable-baselines3

> Requerimientos: https://stable-baselines3.readthedocs.io/en/master/guide/install.html#prerequisites

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from tqdm import tqdm
from IPython.display import Video

import torch

# DQN es la clase del algoritmo en SB3. Todos los algoritmos se importan igual
# desde stable_baselines3 (PPO, A2C, SAC, TD3, DDPG) y comparten la misma API.
from stable_baselines3 import DQN

## Definir constantes y funciones auxiliares

In [ ]:
import os

if os.name == 'posix' and os.uname().sysname == 'Darwin':
    # Set the path to ffmpeg for macOS, replace with your actual path
    os.environ["IMAGEIO_FFMPEG_EXE"] = "/opt/homebrew/bin/ffmpeg"

In [ ]:
DEVICE = 'cpu'  # por defecto, usamos la CPU
if torch.cuda.is_available():  # si hay una GPU disponible (y cuda está instalado)
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():  # si hay un MPS disponible (Metal Performance Shaders)
    DEVICE = 'mps'

In [ ]:
ENV_NAME = "LunarLander-v3" # https://gymnasium.farama.org/environments/box2d/lunar_lander/

In [ ]:
def get_env(env_name, record_video=False, record_every=1, folder="./videos" ):
    """
    Create a Gymnasium environment with optional video recording.
    Args:
        env_name (str): Name of the environment to create.
        record_video (bool): Whether to record video of the episodes.
        record_every (int): Frequency of recording episodes.
        folder (str): Folder to save the recorded videos.
    Returns:
        env (gym.Env): The requested environment.
        
    See also:
        https://gymnasium.farama.org/introduction/record_agent/
    """
    # Initialise the environment
    env = gym.make(env_name, render_mode="rgb_array")

    if record_video:
        env = RecordVideo(env, video_folder=folder,
                    episode_trigger=lambda x: x % record_every == 0,
                    name_prefix=env_name)
    
    return env

In [ ]:
env = get_env(ENV_NAME, record_video=True, record_every=1, folder="./videos/random")

print(f"Environment: {ENV_NAME}")
print(f"Observation space: {env.observation_space}")
print(f"Observation space shape: {env.observation_space.shape}")
print(f"Action space: {env.action_space}")

In [ ]:
video_folder = "./videos/random"
env = get_env(ENV_NAME, record_video=True, record_every=1, folder=video_folder)

obs, info = env.reset()
episode_over = False
while not episode_over:
    action = env.action_space.sample()  # Random action
    obs, reward, terminated, truncated, info = env.step(action)
    episode_over = terminated or truncated

env.close()

In [ ]:
# Ruta al archivo de vídeo en tu sistema de ficheros
video_path = f"{video_folder}/{ENV_NAME}-episode-0.mp4"

# Muestra el vídeo
Video(video_path, embed=True, width=600)

## ¿Qué algoritmo elijo?

SB3 implementa varios algoritmos y **no todos sirven para cualquier entorno**. La primera pregunta es casi siempre:

> **¿El espacio de acciones es discreto o continuo?**

En lo impreso arriba, `LunarLander-v3` tiene `action_space = Discrete(4)`: cuatro acciones posibles (no hacer nada, motor izquierdo, principal, derecho). Existe también `LunarLanderContinuous-v3`, donde las acciones son continuas (`Box`) y se controla la *potencia* de los motores.

### Algoritmos disponibles en SB3

| Algoritmo | Familia | Acciones discretas | Acciones continuas |
|---|---|:---:|:---:|
| **DQN**  | value-based, *off-policy*   | Sí | No |
| **PPO**  | actor-critic, *on-policy*   | Sí | Sí |
| **A2C**  | actor-critic, *on-policy*   | Sí | Sí |
| **SAC**  | actor-critic, *off-policy*  | No | Sí |
| **TD3**  | actor-critic, *off-policy*  | No | Sí |
| **DDPG** | actor-critic, *off-policy*  | No | Sí |

En **[SB3-Contrib](https://sb3-contrib.readthedocs.io/)** hay opciones adicionales: **QR-DQN** (discreto, DQN distribucional), **TQC** (continuo), **RecurrentPPO** (PPO con LSTM, para entornos con estado oculto), **MaskablePPO** (enmascarado de acciones inválidas), **TRPO**, entre otros.

Antes de elegir, conviene consultar la tabla *"Can I use?"* de cada algoritmo en la documentación; por ejemplo, la de [DQN](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html#can-i-use).

### Reglas prácticas

- **Acciones discretas**: comenzar con **DQN** o **PPO**.
- **Acciones continuas**: **SAC** o **TD3** (suelen rendir mejor), o **PPO** como alternativa robusta.
- **Caso de duda**: **PPO** es la opción más general; funciona en casi todos los espacios de acción y es difícil de desestabilizar.

### El argumento `'MlpPolicy'`: la *policy*

El primer argumento del modelo es la **red neuronal** del agente, que se elige según **el tipo de observación**:

- **`MlpPolicy`**: observaciones como **vector** de features (nuestro caso: `Box(8,)`).
- **`CnnPolicy`**: observaciones como **imágenes** (por ejemplo, los píxeles de un juego de Atari).
- **`MultiInputPolicy`**: observaciones tipo **diccionario** (`Dict`), por ejemplo imagen más vector de estado.

## Entrenamiento y evaluación con DQN

Como `LunarLander-v3` tiene acciones **discretas**, podemos usar **DQN** (el algoritmo que implementamos en clase).

El flujo con SB3 es siempre el mismo:

1. **Crear** el modelo: `model = DQN('MlpPolicy', env, ...)`
2. **Entrenar**: `model.learn(total_timesteps=...)`
3. **Evaluar / usar**: `action, _ = model.predict(obs, deterministic=True)`

Primero definimos los hiperparámetros. Cada algoritmo tiene los suyos; los de DQN están documentados [aquí](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html), y conviene verificar la compatibilidad con el entorno en la tabla [*Can I use?*](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html#can-i-use).

In [ ]:
# 'MlpPolicy': red densa (MLP) que provee SB3, para observaciones vectoriales.
# Alternativas: 'CnnPolicy' (imágenes) y 'MultiInputPolicy' (observaciones Dict).
MODEL = 'MlpPolicy'

LR = 5e-4              # Tasa de aprendizaje (learning rate)
BUFFER_SIZE = 100_000  # Tamaño del replay buffer (DQN es off-policy: reusa experiencia)
BATCH_SIZE = 32       # Tamaño del minibatch
GAMMA = 0.99          # Factor de descuento

TRAIN_STEPS = 500_000  # Pasos totales de entrenamiento (interacciones con el entorno)
LOG_INTERVAL = 10      # Cada cuántos episodios se imprimen las métricas

In [ ]:
env = get_env(ENV_NAME, record_video=True, record_every=100, folder="./videos/DQN/training")

# Construcción del agente: DQN(policy, env, **hiperparámetros).
#   - 'MlpPolicy': arquitectura de la red (definida arriba).
#   - buffer_size: tamaño del replay buffer (propio de algoritmos off-policy como DQN).
#   - device: 'cpu' / 'cuda' / 'mps'.
#   - verbose=1: imprime las métricas durante el entrenamiento (rollout/ time/ train/).
model = DQN('MlpPolicy', env, learning_rate=LR, buffer_size=BUFFER_SIZE, batch_size=BATCH_SIZE, gamma=GAMMA, device=DEVICE, verbose=1)

# .learn() ejecuta todo el loop de entrenamiento (interacción + actualización de la red).
#   - total_timesteps: cantidad total de pasos en el entorno.
#   - progress_bar: muestra una barra de progreso.
model.learn(total_timesteps=TRAIN_STEPS, log_interval=LOG_INTERVAL, progress_bar=True)

env.close()

Durante el entrenamiento, SB3 imprime métricas agrupadas en tres bloques:

**`rollout/`** (interacción con el entorno)
- **`ep_len_mean`**: longitud media del episodio (promedio de los últimos ~100).
- **`ep_rew_mean`**: recompensa media por episodio (promedio de los últimos ~100).
- **`exploration_rate`**: tasa ε de exploración en DQN (fracción de acciones aleatorias en epsilon-greedy).

**`time/`** (rendimiento del proceso)
- **`episodes`**: número total de episodios jugados.
- **`fps`**: frames por segundo (incluye actualizaciones de gradiente).
- **`time_elapsed`**: segundos transcurridos desde el inicio del entrenamiento.
- **`total_timesteps`**: total de pasos dados en el entorno.

**`train/`** (optimización del modelo)
- **`learning_rate`**: tasa de aprendizaje usada actualmente.
- **`loss`**: pérdida total (función objetivo de la red).
- **`n_updates`**: cantidad de actualizaciones realizadas al modelo.

Documentación: https://stable-baselines3.readthedocs.io/en/master/common/logger.html

In [ ]:
env = get_env(ENV_NAME, record_video=True, record_every=1, folder="./videos/DQN/eval")

for i in tqdm(range(5), desc="Evaluating DQN model"):
    obs, info = env.reset()
    done = False
    while not done:
        # model.predict devuelve (acción, estado_interno). El segundo valor solo
        # se usa con políticas recurrentes (LSTM); por eso lo descartamos con _.
        # deterministic=True: toma la mejor acción (sin exploración), ideal para evaluar.
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

env.close()

In [ ]:
from IPython.display import display, Video

video_paths = [
    f"./videos/DQN/eval/{ENV_NAME}-episode-{i}.mp4" for i in range(5)
]

# Display all videos
for path in video_paths:
    display(Video(path, embed=True, width=600))


### Paralelizar el entrenamiento (entornos vectorizados)

SB3 puede ejecutar **varias copias del entorno en paralelo** (*vectorized environments*) y recolectar experiencia de todas a la vez. Esto acelera la recolección de datos, especialmente en algoritmos *on-policy* como PPO o A2C.

- **`make_vec_env(...)`**: helper para crear N entornos en una sola llamada.
- **`SubprocVecEnv`**: ejecuta cada entorno en un **proceso** separado (aprovecha múltiples CPU; conviene cuando el entorno es costoso).
- **`DummyVecEnv`**: ejecuta los entornos en el **mismo proceso** (menos overhead; conviene cuando el entorno es liviano).

> Al usar `SubprocVecEnv` en un script `.py`, el código de entrenamiento debe ir dentro de `if __name__ == "__main__":` (en notebooks no hace falta).

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import SubprocVecEnv   # cada entorno en su propio proceso
from stable_baselines3.common.env_util import make_vec_env   # helper para crear N entornos

N_ENVS = 8

# make_vec_env crea N copias del entorno y las agrupa en un VecEnv.
# vec_env_cls=SubprocVecEnv las ejecuta en paralelo (un proceso por entorno).
vec_env = make_vec_env(ENV_NAME, n_envs=N_ENVS, seed=42, vec_env_cls=SubprocVecEnv)

# El modelo es el mismo, pero ahora recibe un VecEnv en lugar de un entorno simple:
# SB3 maneja la vectorización de forma transparente.
model = DQN(
    'MlpPolicy', 
    vec_env, 
    learning_rate=LR,
    buffer_size=BUFFER_SIZE,
    batch_size=BATCH_SIZE,
    train_freq=1,   # cada cuántos pasos se actualiza la red
    gamma=GAMMA,
    device=DEVICE,
    verbose=1
)

# Entrenamiento (misma llamada .learn() que en el caso de un solo entorno)
model.learn(total_timesteps=TRAIN_STEPS, log_interval=LOG_INTERVAL, progress_bar=True)


In [ ]:
env = get_env(ENV_NAME, record_video=True, record_every=1, folder="./videos/multiDQN/eval")

for i in tqdm(range(5), desc="Evaluating DQN model"):
    obs, info = env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

env.close()

video_paths = [
    f"./videos/multiDQN/eval/{ENV_NAME}-episode-{i}.mp4" for i in range(5)
]

# Display all videos
for path in video_paths:
    display(Video(path, embed=True, width=600))